# Model Evaluation — BERT AG News Classifier
Detailed evaluation of the fine-tuned model on the held-out test set.

**Sections**
1. Load model & test data  
2. Generate predictions  
3. Accuracy & F1 scores  
4. Classification report  
5. Confusion matrix  
6. Confidence analysis  
7. Error analysis  


## 1 · Imports & Load Model

In [8]:
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
print(os.listdir(MODEL_DIR))
import seaborn as sns
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import BertTokenizerFast, BertForSequenceClassification
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

MODEL_DIR   = "models/bert_ag_news"
MAX_LENGTH  = 128
BATCH_SIZE  = 64
LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
PALETTE     = ["#3B82F6", "#10B981", "#F59E0B", "#8B5CF6"]

Path("outputs").mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


NameError: name 'MODEL_DIR' is not defined

In [3]:
from transformers import BertTokenizerFast, BertForSequenceClassification

tokenizer = BertTokenizerFast.from_pretrained(MODEL_DIR)
model = BertForSequenceClassification.from_pretrained(MODEL_DIR)

model.eval()
model.to(device)


NameError: name 'MODEL_DIR' is not defined

## 2 · Load & Tokenise Test Set

In [7]:
raw_test = load_dataset("ag_news", split="test")

def tok(batch):
    return tokenizer(batch["text"], truncation=True,
                     max_length=MAX_LENGTH, padding="max_length")

test_ds = raw_test.map(tok, batched=True, remove_columns=["text"])
test_ds = test_ds.rename_column("label", "labels")
test_ds.set_format("torch")
print(f"Test set: {len(test_ds):,} examples")


NameError: name 'load_dataset' is not defined

## 3 · Generate Predictions

In [6]:
from tqdm.auto import tqdm

loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
all_logits, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(loader, desc="Predicting"):
        labels = batch.pop("labels").to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

logits = np.concatenate(all_logits)
labels = np.concatenate(all_labels)
probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()
preds  = np.argmax(logits, axis=-1)

print(f"Predictions generated for {len(preds):,} examples ✅")


NameError: name 'DataLoader' is not defined

## 4 · Accuracy & F1 Scores

In [5]:
acc  = accuracy_score(labels, preds)
f1_m = f1_score(labels, preds, average="macro")
f1_w = f1_score(labels, preds, average="weighted")

summary = pd.DataFrame({
    "Metric": ["Accuracy", "F1 (Macro)", "F1 (Weighted)"],
    "Score":  [f"{acc:.4f}", f"{f1_m:.4f}", f"{f1_w:.4f}"],
    "Percentage": [f"{acc*100:.2f}%", f"{f1_m*100:.2f}%", f"{f1_w*100:.2f}%"],
})
display(summary.style.hide(axis="index"))

# Save
with open("outputs/metrics.json", "w") as f:
    json.dump({"accuracy": acc, "f1_macro": f1_m, "f1_weighted": f1_w}, f, indent=2)
print("\nSaved → outputs/metrics.json")


NameError: name 'accuracy_score' is not defined

## 5 · Classification Report

In [4]:
report = classification_report(
    labels, preds,
    target_names=list(LABEL_NAMES.values()),
    digits=4,
)
print(report)


NameError: name 'classification_report' is not defined

## 6 · Visualisation Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle("BERT AG News — Evaluation Dashboard",
             fontsize=18, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# ── Confusion Matrix ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
cm     = confusion_matrix(labels, preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=list(LABEL_NAMES.values()),
            yticklabels=list(LABEL_NAMES.values()),
            linewidths=0.5, ax=ax1, cbar_kws={"label": "%"})
ax1.set_title("Confusion Matrix (%)", fontsize=13, fontweight="bold")
ax1.set_ylabel("True"); ax1.set_xlabel("Predicted")

# ── Per-class F1 ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
f1s  = f1_score(labels, preds, average=None)
bars = ax2.barh(list(LABEL_NAMES.values()), f1s, color=PALETTE, height=0.5)
ax2.set_xlim(0, 1.05); ax2.set_xlabel("F1 Score")
ax2.set_title("Per-Class F1 Score", fontsize=13, fontweight="bold")
for bar, v in zip(bars, f1s):
    ax2.text(v + 0.005, bar.get_y() + bar.get_height()/2,
             f"{v:.4f}", va="center", fontsize=11)
ax2.axvline(np.mean(f1s), color="gray", linestyle="--",
            label=f"Mean: {np.mean(f1s):.4f}")
ax2.legend()

# ── Class distribution ───────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
x, w = np.arange(4), 0.35
true_counts = np.bincount(labels.astype(int), minlength=4)
pred_counts = np.bincount(preds, minlength=4)
ax3.bar(x - w/2, true_counts, w, label="True",      color=PALETTE, alpha=0.7)
ax3.bar(x + w/2, pred_counts, w, label="Predicted", color=PALETTE, alpha=1.0)
ax3.set_xticks(x); ax3.set_xticklabels(list(LABEL_NAMES.values()))
ax3.set_ylabel("Count")
ax3.set_title("True vs Predicted Distribution", fontsize=13, fontweight="bold")
ax3.legend()

# ── Confidence histogram ─────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
max_probs = probs.max(axis=1)
correct   = preds == labels
ax4.hist(max_probs[correct],  bins=40, alpha=0.7, color="#10B981", label="Correct")
ax4.hist(max_probs[~correct], bins=40, alpha=0.7, color="#EF4444", label="Incorrect")
ax4.set_xlabel("Max Softmax Probability"); ax4.set_ylabel("Count")
ax4.set_title("Confidence Distribution", fontsize=13, fontweight="bold")
ax4.legend()

plt.tight_layout()
plt.savefig("outputs/evaluation_report.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/evaluation_report.png")


## 7 · Error Analysis — Hardest Examples

In [ ]:
# Pull original text back for analysis
raw_test = load_dataset("ag_news", split="test")
texts = raw_test["text"]

# Find misclassified examples sorted by confidence of wrong prediction
wrong_idx  = np.where(preds != labels)[0]
wrong_conf = probs.max(axis=1)[wrong_idx]
top_errors = wrong_idx[np.argsort(-wrong_conf)][:15]   # top-15 most confident errors

rows = []
for i in top_errors:
    rows.append({
        "Headline"  : texts[int(i)][:90] + "…",
        "True"      : LABEL_NAMES[int(labels[i])],
        "Predicted" : LABEL_NAMES[int(preds[i])],
        "Confidence": f"{probs[i].max()*100:.1f}%",
    })

err_df = pd.DataFrame(rows)
print(f"Total misclassified: {len(wrong_idx):,} / {len(preds):,}  "
      f"({len(wrong_idx)/len(preds)*100:.2f}%)")
print("\nTop-15 most confident errors:")
display(err_df.style.hide(axis="index"))
